# Lesson 1: Variational inference


## Hyperparameters

All experiment settings are collected below. Edit this panel, then **Run All**.


In [ ]:
# HYPERPARAMETERS — edit here, then Run All

# Objective and target distribution
TEMPERATURE = 0.5          # Entropy weight T; try 0.0, 0.2, 0.5, or 2.0.
NUM_MODES = 40             # Number of equally weighted Gaussian target modes.
MODE_BOUND = 40.0          # Mode centers lie in [-MODE_BOUND, MODE_BOUND]^2.
TARGET_VARIANCE = 1.0      # Variance of each target component in each coordinate.
TARGET_SEED = 0            # Seed that fixes the target's component locations.

# Gaussian policy
HIDDEN_SIZE = 32           # Units in each of the two hidden layers.
INITIAL_MEAN = (0.0, 0.0)  # Initial mean of the two-dimensional policy.
INITIAL_STD = 30.0         # Initial standard deviation in both coordinates.

# Optimization (both gradient estimators use these settings)
SEED = 7                  # Shared seed for policy initialization and training.
TRAINING_STEPS = 800       # Number of optimizer updates per estimator.
BATCH_SIZE = 512           # Monte Carlo samples per gradient estimate (at least 2).
LEARNING_RATE = 3e-3       # Adam learning rate for policy parameters.
MAX_GRAD_NORM = 10.0       # Upper bound on the norm of an optimizer gradient.

# Evaluation and animation
LOG_EVERY = 20             # Record diagnostics every this many updates.
NUM_EVAL_SAMPLES = 4096    # Samples used to estimate the return and scaled KL.
NUM_GIF_SAMPLES = 160      # Number of fixed-noise particles in each GIF panel.
ANIMATION_SEED = None      # Integer for separate GIF noise; None uses training RNG.
GIF_FPS = 2.0              # Saved animation frames per second.
GIF_DPI = 100              # Saved animation resolution in dots per inch.
PLOT_BOUND = 56.0          # Plot both coordinates over [-PLOT_BOUND, PLOT_BOUND].
GRID_RESOLUTION = 320      # Grid points per coordinate for contours and normalization.
CONTOUR_LEVELS = 80        # Number of reward and target-density contour levels.
REWARD_PLOT_FLOOR = -1000.0  # Lowest reward shown in the contour plot.


## Problem setting

We fit a diagonal two-dimensional Gaussian $q_\theta$ to a reward-induced target distribution by minimizing reverse KL, following Lesson 1 of the slides.

$$p_T(x)=\frac{\exp[R(x)/T]}{Z_T}, \qquad T>0,$$

where $T$ is the temperature and $Z_T$ is the normalizing constant. The reward is the log density of an equal-weight mixture of 40 Gaussians (GMM-40). All reward, policy, training, and plotting helpers are provided in this notebook.

Your task is to implement two variational-inference losses: **reparameterization** and **log derivative**. Optional hints above each exercise name the relevant PyTorch calls. After implementing the losses, vary the temperature to explore the balance between reward and entropy, including pure reward maximization at $T=0$.


## Reverse KL and the entropy-regularized objective

For $T>0$, multiply the reverse KL from $q_\theta$ to $p_T$ by the temperature:

$$\begin{aligned}
T\,\mathrm{KL}(q_\theta\,\|\,p_T)
&=T\,\mathbb E_{q_\theta}[\log q_\theta(X)-\log p_T(X)]\\
&=-\mathbb E_{q_\theta}[R(X)]-T\,\mathcal H(q_\theta)+T\log Z_T.
\end{aligned}$$

The term $T\log Z_T$ is constant with respect to $\theta$, so minimizing reverse KL is equivalent to maximizing

$$\boxed{J_T(\theta)=\mathbb E_{q_\theta}[R(X)]+T\,\mathcal H(q_\theta).}$$

Unlike the unscaled KL, this expression can be evaluated directly at zero temperature: $J_0(\theta)=\mathbb E[R(X)]$, which recovers ordinary reward maximization within this same lesson. PyTorch minimizes the loss $-J_T$.

## Setup

Follow `../README.md`, then select the project's `.venv` as the notebook kernel.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import animation
import torch
from torch import nn
from torch.distributions import Independent, Normal

torch.manual_seed(SEED)
plt.rcParams["figure.figsize"] = (7, 4)

## Temperature experiments

Start with $T=0.5$ to match the slides. After implementing both losses, try $T=0$, $0.2$, and $2$ to compare pure reward maximization with entropy-regularized variational inference.

Change `TEMPERATURE` below and rerun this cell and all following cells (or use Run All). This refreshes the target plot, diagnostics, training, and animation together. Keep the random seed fixed when comparing temperatures.

In [ ]:
if not math.isfinite(TEMPERATURE) or TEMPERATURE < 0.0:
    raise ValueError("Temperature must be finite and non-negative.")

## 1. Reward and Boltzmann target

We use the seed-0 GMM-40 benchmark. Its 40 means are sampled uniformly from $[-40,40]^2$, every component has covariance $I$, and all components have equal weight. We define $R(x)=\log p_{\mathrm{ref}}(x)$ from this mixture. At $T=1$, the Boltzmann target equals the reference mixture; changing $T$ changes its concentration.

The training losses do not need the normalizing constant. The provided grid calculation approximates it only for visualization and diagnostics.

In [ ]:
TARGET_STD = math.sqrt(TARGET_VARIANCE)
mode_generator = torch.Generator().manual_seed(TARGET_SEED)
MODE_LOCATIONS = -MODE_BOUND + 2.0 * MODE_BOUND * torch.rand(
    (NUM_MODES, 2), generator=mode_generator
)
LOG_MIXTURE_WEIGHT = -math.log(NUM_MODES)


def target_reward(states: torch.Tensor) -> torch.Tensor:
    """Log-density reward of the configured equal-weight GMM."""
    standardized = (states.unsqueeze(-2) - MODE_LOCATIONS) / TARGET_STD
    component_log_prob = (
        -0.5 * standardized.square().sum(dim=-1)
        - 2.0 * math.log(TARGET_STD * math.sqrt(2.0 * math.pi))
        + LOG_MIXTURE_WEIGHT
    )
    return torch.logsumexp(component_log_prob, dim=-1)


def boltzmann_target_on_grid(
    rewards: torch.Tensor, x_axis: torch.Tensor, y_axis: torch.Tensor, temperature: float
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return a normalized grid density and T * log(Z_T)."""
    if temperature < 0.0:
        raise ValueError("Temperature must be non-negative.")

    maximum_reward = rewards.max()
    if temperature == 0.0:
        # Plot the limiting point mass as a narrow grid spike.
        maximum_mask = torch.isclose(rewards, maximum_reward).to(rewards.dtype)
        partition = torch.trapezoid(
            torch.trapezoid(maximum_mask, x_axis, dim=1), y_axis, dim=0
        )
        density = maximum_mask / partition
        scaled_log_partition = maximum_reward
    else:
        # Subtract the maximum reward before exponentiating for stability.
        shifted_weights = torch.exp((rewards - maximum_reward) / temperature)
        shifted_partition = torch.trapezoid(
            torch.trapezoid(shifted_weights, x_axis, dim=1), y_axis, dim=0
        )
        density = shifted_weights / shifted_partition
        scaled_log_partition = temperature * shifted_partition.log() + maximum_reward

    return density, scaled_log_partition


plot_axis = torch.linspace(-PLOT_BOUND, PLOT_BOUND, GRID_RESOLUTION)
grid_x, grid_y = torch.meshgrid(plot_axis, plot_axis, indexing="xy")
grid_points = torch.stack((grid_x, grid_y), dim=-1)
reward_on_grid = target_reward(grid_points)
boltzmann_density, SCALED_LOG_Z_T = boltzmann_target_on_grid(
    reward_on_grid, plot_axis, plot_axis, TEMPERATURE
)
display_reward = reward_on_grid.clamp(min=REWARD_PLOT_FLOOR)
density_levels = torch.linspace(0.0, boltzmann_density.max().item(), CONTOUR_LEVELS)
reward_levels = torch.linspace(display_reward.min().item(), display_reward.max().item(), CONTOUR_LEVELS)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
reward_plot = axes[0].contour(grid_x, grid_y, display_reward, levels=reward_levels, cmap="viridis", linewidths=0.7, alpha=0.9)
density_plot = axes[1].contour(grid_x, grid_y, boltzmann_density, levels=density_levels, cmap="viridis", linewidths=0.7, alpha=0.9)
for axis in axes:
    axis.scatter(
        MODE_LOCATIONS[:, 0], MODE_LOCATIONS[:, 1],
        s=12, color="tab:blue", edgecolors="none", zorder=3,
    )
    axis.set(
        xlim=(-PLOT_BOUND, PLOT_BOUND), ylim=(-PLOT_BOUND, PLOT_BOUND),
        xticks=(-MODE_BOUND, 0, MODE_BOUND), yticks=(-MODE_BOUND, 0, MODE_BOUND),
        xlabel="state x₁", ylabel="state x₂", aspect="equal",
    )
axes[0].set_title(f"GMM-{NUM_MODES} reward R(x)")
axes[1].set_title(f"Boltzmann target (T={TEMPERATURE:g})")
plt.tight_layout()
plt.show()

print(f"Temperature-scaled log partition T log(Z_T) = {SCALED_LOG_Z_T.item():.4f}")

## 2. Gaussian variational policy

The network receives a constant observation and produces two means and two log standard deviations. Exponentiating the log standard deviations makes them positive. `Independent(Normal(...), 1)` treats the two coordinates as a single 2D event.

A single Gaussian cannot efficiently cover all 40 separated modes under reverse KL, so it will normally select one mode. At positive temperature, the entropy term favors finite standard deviations. At zero temperature, the objective favors concentrating near a reward maximum. The network's `log_std` output is not clipped.

In [ ]:
class GaussianPolicy(nn.Module):
    """Neural network that parameterizes a diagonal 2D Gaussian."""

    def __init__(self, hidden_size: int = HIDDEN_SIZE):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(1, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
        )
        self.mean_head = nn.Linear(hidden_size, 2)
        self.log_std_head = nn.Linear(hidden_size, 2)

        # Begin with a broad cloud spanning much of the GMM-40 landscape.
        nn.init.zeros_(self.mean_head.weight)
        with torch.no_grad():
            self.mean_head.bias.copy_(torch.tensor(INITIAL_MEAN))
        nn.init.zeros_(self.log_std_head.weight)
        nn.init.constant_(self.log_std_head.bias, math.log(INITIAL_STD))

    def forward(self) -> tuple[torch.Tensor, torch.Tensor]:
        device = next(self.parameters()).device
        constant_observation = torch.ones(1, 1, device=device)
        features = self.backbone(constant_observation)

        mean = self.mean_head(features).squeeze(0)
        log_std = self.log_std_head(features).squeeze(0)
        return mean, log_std

    def distribution(self) -> Independent:
        mean, log_std = self()
        return Independent(Normal(mean, log_std.exp()), 1)


policy = GaussianPolicy()
initial_mean, initial_log_std = policy()
print(f"Initial mean: {initial_mean.tolist()}")
print(f"Initial std:  {initial_log_std.exp().tolist()}")

## 3. Loss A: reparameterization estimator

Move the randomness into parameter-free Gaussian noise:

$$\varepsilon_i\sim\mathcal N(0,I_2),\qquad X_i=\mu_\theta+\sigma_\theta\odot\varepsilon_i.$$

The VI objective can then be written as $J_T=\mathbb E_\varepsilon[R(X)-T\log q_\theta(X)]$. For a Gaussian, we can evaluate the entropy analytically and estimate only the reward term:

$$L_{\mathrm{path}}=-\frac{1}{N}\sum_i R(X_i)-T\,\mathcal H(q_\theta)=-\widehat J_T.$$

**Task:** implement this scalar loss using differentiable samples and the policy entropy. Use the function's `temperature` argument so the same implementation works for every experiment.

**Optional hints** — click a label to expand.

<details>
<summary><strong>Get a hint: sampling</strong></summary>
<p>Get the distribution with <code>q = policy.distribution()</code>. Draw differentiable samples with <code>q.rsample((num_samples,))</code>; the result has shape <code>(num_samples, 2)</code>. This uses the reparameterization trick. Keep the samples attached so gradients can flow into the policy.</p>
</details>

<details>
<summary><strong>Get a hint: rewards and entropy</strong></summary>
<p>Call <code>target_reward(samples)</code> for rewards of shape <code>(num_samples,)</code>. The Gaussian distribution provides its analytic entropy through <code>q.entropy()</code>, a scalar that already includes both state coordinates.</p>
</details>

<details>
<summary><strong>Get a hint: the VI loss and temperature</strong></summary>
<p>Average the rewards with <code>.mean()</code>. Combine negative mean reward with negative <code>temperature</code> times the entropy, keeping both terms attached to the computation graph. At zero temperature, the entropy term vanishes and this becomes ordinary reward maximization.</p>
</details>

In [ ]:
def reparameterization_maxent_loss(
    policy: GaussianPolicy, num_samples: int, temperature: float
) -> torch.Tensor:
    """Negative pathwise estimate of the VI objective J_T."""
    # TODO: Implement the pathwise VI loss. Optional hints are above.
    raise NotImplementedError("Implement the pathwise VI loss")

## 4. Loss B: log-derivative estimator

Write the same variational-inference objective as

$$J_T=\mathbb E_{X\sim q_\theta}[R(X)-T\log q_\theta(X)].$$

The score-function identity gives

$$\nabla_\theta J_T=\mathbb E_{X\sim q_\theta}\left[(R(X)-T\log q_\theta(X)-b)\,\nabla_\theta\log q_\theta(X)\right].$$

The extra constant $-T$ from differentiating the log density inside the expectation can be omitted because $\mathbb E_q[\nabla_\theta\log q_\theta]=0$.

Use a **leave-one-out baseline**: for each sample, subtract the mean learning signal of all other samples. This baseline is independent of the current sample and reduces variance without biasing the estimator. Treat the sampled states and the centered signal as constants during differentiation.

**Task:** implement a negative score-function surrogate using the temperature-dependent learning signal and a leave-one-out baseline. Its gradient should make gradient descent increase $J_T$.

**Optional hints** — click a label to expand.

<details>
<summary><strong>Get a hint: sampling</strong></summary>
<p>Get the distribution with <code>q = policy.distribution()</code>. You can reuse reparameterized sampling with <code>q.rsample((num_samples,)).detach()</code>, which returns shape <code>(num_samples, 2)</code>. Detach before evaluating rewards or log probabilities to remove the pathwise gradient. Alternatively, <code>q.sample((num_samples,))</code> already gives samples without a gradient path.</p>
</details>

<details>
<summary><strong>Get a hint: rewards, log probabilities, and the baseline</strong></summary>
<p>Use <code>target_reward(samples)</code> for rewards and <code>q.log_prob(samples)</code> for log probabilities. Both have shape <code>(num_samples,)</code>; log probabilities already sum over the two state coordinates. Form <code>signal = rewards - temperature * log_probabilities</code>. The leave-one-out baseline is <code>(signal.sum() - signal) / (num_samples - 1)</code>. Require at least two samples.</p>
</details>

<details>
<summary><strong>Get a hint: gradients and the surrogate loss</strong></summary>
<p>Use <code>(signal - baseline).detach()</code> for the centered learning signal. Multiply it by the attached log probabilities, take <code>.mean()</code>, and negate the result. Only the log probabilities outside the detached signal supply gradients. At zero temperature, the signal reduces to the reward.</p>
</details>

In [ ]:
def log_derivative_maxent_loss(
    policy: GaussianPolicy, num_samples: int, temperature: float
) -> torch.Tensor:
    """Negative score-function surrogate for the VI objective J_T."""
    # TODO: Implement the log-derivative VI loss. Optional hints are above.
    raise NotImplementedError("Implement the log-derivative VI loss")

## 5. Check the implementations

These structural checks verify that each loss is a finite scalar and supplies gradients to all policy parameters.

In [ ]:
def check_loss_function(loss_function) -> None:
    torch.manual_seed(0)
    test_policy = GaussianPolicy()
    loss = loss_function(test_policy, num_samples=128, temperature=TEMPERATURE)

    assert loss.ndim == 0, "The loss must be a scalar."
    assert torch.isfinite(loss), "The loss must be finite."
    loss.backward()

    gradients = [parameter.grad for parameter in test_policy.parameters()]
    assert all(gradient is not None for gradient in gradients)
    assert all(torch.isfinite(gradient).all() for gradient in gradients)
    print(f"{loss_function.__name__}: check passed")


check_loss_function(reparameterization_maxent_loss)
check_loss_function(log_derivative_maxent_loss)

## 6. Train both policies

For diagnostics we estimate both the MaxEnt return $J_T$ (which should increase) and $T\,\mathrm{KL}(q\|p_T)$ (which should decrease). At $T=0$, the latter is the reward gap $\max_x R(x)-\mathbb E[R]$. The score-function surrogate value is not itself either diagnostic; only its gradient is useful.

In [ ]:
def policy_statistics(policy: GaussianPolicy) -> tuple[torch.Tensor, torch.Tensor]:
    with torch.no_grad():
        mean, log_std = policy()
    return mean.cpu(), log_std.exp().cpu()


def estimate_temperature_scaled_kl(
    policy: GaussianPolicy, num_samples: int = NUM_EVAL_SAMPLES
) -> float:
    with torch.no_grad():
        distribution = policy.distribution()
        states = distribution.sample((num_samples,))
        scaled_log_density_ratio = (
            -target_reward(states)
            + TEMPERATURE * distribution.log_prob(states)
            + SCALED_LOG_Z_T
        )
    return scaled_log_density_ratio.mean().item()


def estimate_maxent_return(policy: GaussianPolicy, num_samples: int = NUM_EVAL_SAMPLES) -> float:
    with torch.no_grad():
        distribution = policy.distribution()
        mean_reward = target_reward(distribution.sample((num_samples,))).mean()
        return (mean_reward + TEMPERATURE * distribution.entropy()).item()


def train_policy(
    loss_function,
    *,
    seed: int = SEED,
    steps: int = TRAINING_STEPS,
    batch_size: int = BATCH_SIZE,
    learning_rate: float = LEARNING_RATE,
    log_every: int = LOG_EVERY,
):
    torch.manual_seed(seed)
    policy = GaussianPolicy()
    optimizer = torch.optim.Adam(policy.parameters(), lr=learning_rate)
    # Reusing these noise vectors makes individual dots move smoothly in the GIF.
    animation_generator = (None if ANIMATION_SEED is None else
                           torch.Generator().manual_seed(ANIMATION_SEED))
    animation_noise = torch.randn(NUM_GIF_SAMPLES, 2, generator=animation_generator)
    history = {
        "step": [], "mean": [], "std": [],
        "scaled_kl": [], "maxent_return": [], "samples": [],
    }

    for step in range(steps + 1):
        if step % log_every == 0 or step == steps:
            mean, std = policy_statistics(policy)
            history["step"].append(step)
            history["mean"].append(mean)
            history["std"].append(std)
            history["scaled_kl"].append(estimate_temperature_scaled_kl(policy))
            history["maxent_return"].append(estimate_maxent_return(policy))
            history["samples"].append(mean + std * animation_noise)

        if step == steps:
            break

        optimizer.zero_grad()
        loss = loss_function(policy, batch_size, temperature=TEMPERATURE)
        loss.backward()
        nn.utils.clip_grad_norm_(policy.parameters(), max_norm=MAX_GRAD_NORM)
        optimizer.step()

    return policy, history

In [ ]:
pathwise_policy, pathwise_history = train_policy(reparameterization_maxent_loss)
score_policy, score_history = train_policy(log_derivative_maxent_loss)

for name, trained_policy in [
    ("Reparameterization", pathwise_policy),
    ("Log derivative", score_policy),
]:
    mean, std = policy_statistics(trained_policy)
    scaled_kl = estimate_temperature_scaled_kl(trained_policy)
    maxent_return = estimate_maxent_return(trained_policy)
    print(
        f"{name:20s} -> mean = {mean.numpy().round(3)}, "
        f"std = {std.numpy().round(3)}, J_T = {maxent_return:.3f}, T*KL = {scaled_kl:.3f}"
    )

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.ravel()

for axis, name, trained_policy, color in [
    (axes[0], "Reparameterization", pathwise_policy, "tab:blue"),
    (axes[1], "Log derivative", score_policy, "tab:orange"),
]:
    axis.contour(grid_x, grid_y, display_reward, levels=reward_levels, cmap="viridis", linewidths=0.7, alpha=0.9)
    with torch.no_grad():
        learned_log_density = trained_policy.distribution().log_prob(grid_points)
    peak = learned_log_density.max().item()
    axis.contour(
        grid_x, grid_y, learned_log_density,
        levels=[peak - 4.5, peak - 2.0, peak - 0.5], colors=color, linewidths=2,
    )
    mean, _ = policy_statistics(trained_policy)
    axis.scatter(*mean, marker="*", s=160, color=color, edgecolor="white")
    axis.set(title=f"{name}: final policy", xlim=(-PLOT_BOUND, PLOT_BOUND), ylim=(-PLOT_BOUND, PLOT_BOUND), xticks=(-MODE_BOUND, 0, MODE_BOUND), yticks=(-MODE_BOUND, 0, MODE_BOUND), xlabel="state x₁", ylabel="state x₂", aspect="equal")

axes[2].plot(pathwise_history["step"], pathwise_history["maxent_return"], label="reparameterization")
axes[2].plot(score_history["step"], score_history["maxent_return"], label="log derivative")
axes[2].set(title="Maximum-entropy reward", xlabel="optimization step", ylabel="estimated J_T")
axes[2].legend()

for history, name, color in [
    (pathwise_history, "reparameterization", "tab:blue"),
    (score_history, "log derivative", "tab:orange"),
]:
    std_values = torch.stack(history["std"])
    axes[3].plot(history["step"], std_values[:, 0], color=color, label=f"{name} σ₁")
    axes[3].plot(history["step"], std_values[:, 1], color=color, linestyle="--", label=f"{name} σ₂")
axes[3].axhline(TARGET_STD * math.sqrt(TEMPERATURE), color="black", linestyle=":", label="local target std")
axes[3].set(title="Policy standard deviations", xlabel="optimization step", ylabel="std")
axes[3].legend(fontsize=8)

for axis in axes:
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Discussion and temperature experiments

After implementing both losses, change `TEMPERATURE` in the hyperparameter panel and use **Run All** (or use Run All). The helpers pass the current temperature explicitly to both losses. Keeping the seed fixed makes comparisons easier.

1. Compare $T=0$, $T=0.2$, $T=0.5$, and $T=2$. How do the target density and learned standard deviations change?
2. At $T=0$, both losses optimize ordinary expected reward. Why does removing entropy encourage variance collapse?
3. Why can $T\log Z_T$ be ignored during optimization but not when reporting the temperature-scaled KL value?
4. Reverse KL is mode seeking. How do the broad initialization and stochastic gradient noise affect which of the 40 modes is selected? Does raising the temperature let a single Gaussian represent every mode?
5. Remove the leave-one-out baseline from the log-derivative estimator and compare the learning curve.

## 7. Animate the learned samples

Each dot uses the same fixed noise vector in every frame. Its motion therefore shows how the learned mean and standard deviations transform the policy's sample cloud over the GMM-40 landscape. The loss panel shows the optimized MaxEnt loss $-J_T$; its dashed vertical line and point markers track the current frame.

In [ ]:
def save_sample_animation(histories, output_path: Path) -> animation.FuncAnimation:
    fig, axes = plt.subplots(
        1, 3, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1, 1, 0.9]}
    )
    landscape_axes = axes[:2]
    loss_axis = axes[2]
    sample_artists = []
    trail_artists = []

    for axis, (name, history, color) in zip(landscape_axes, histories):
        axis.contour(grid_x, grid_y, display_reward, levels=reward_levels, cmap="viridis", linewidths=0.7, alpha=0.9)
        samples = history["samples"][0]
        sample_artists.append(
            axis.scatter(samples[:, 0], samples[:, 1], s=16, alpha=0.55, color=color, zorder=4)
        )
        trail_artists.append(axis.plot([], [], color=color, linewidth=2, zorder=3)[0])
        axis.set(xlim=(-PLOT_BOUND, PLOT_BOUND), ylim=(-PLOT_BOUND, PLOT_BOUND), xticks=(-MODE_BOUND, 0, MODE_BOUND), yticks=(-MODE_BOUND, 0, MODE_BOUND), xlabel="state x₁", ylabel="state x₂", aspect="equal")
        axis.set_title(name)

    curve_colors = ("tab:blue", "tab:orange")
    loss_values = []
    loss_markers = []
    for (name, history, _), curve_color in zip(histories, curve_colors):
        values = [-value for value in history["maxent_return"]]
        loss_values.append(values)
        loss_axis.plot(history["step"], values, color=curve_color, linewidth=2, label=name)
        loss_markers.append(
            loss_axis.scatter([history["step"][0]], [values[0]], s=34, color=curve_color, zorder=3)
        )
    current_step_line = loss_axis.axvline(
        histories[0][1]["step"][0], color="#1F2937", linestyle="--", linewidth=1.4
    )
    loss_axis.set(
        title="Maximum-entropy loss", xlabel="optimization step",
        ylabel="−Jₜ",
        xlim=(histories[0][1]["step"][0], histories[0][1]["step"][-1]),
    )
    loss_axis.grid(alpha=0.2)
    loss_axis.legend(fontsize=8)

    step_label = fig.suptitle("")

    def update(frame):
        for (_, history, _), samples_artist, trail_artist in zip(
            histories, sample_artists, trail_artists
        ):
            samples_artist.set_offsets(history["samples"][frame])
            means = torch.stack(history["mean"][: frame + 1])
            trail_artist.set_data(means[:, 0], means[:, 1])
        current_step = histories[0][1]["step"][frame]
        current_step_line.set_xdata([current_step, current_step])
        for values, marker in zip(loss_values, loss_markers):
            marker.set_offsets([[current_step, values[frame]]])
        step_label.set_text(
            f"Maximum-entropy policy samples (T={TEMPERATURE:g}) — "
            f"training step {current_step}"
        )
        return [*sample_artists, *trail_artists, *loss_markers, current_step_line, step_label]

    sample_animation = animation.FuncAnimation(
        fig, update, frames=len(histories[0][1]["step"]), interval=1000 / GIF_FPS,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    sample_animation.save(output_path, writer=animation.PillowWriter(fps=GIF_FPS), dpi=GIF_DPI)
    plt.close(fig)
    return sample_animation


output_directory = Path("L1-VariationalInference")
if not output_directory.is_dir():
    output_directory = Path(".")
GIF_PATH = output_directory / "maximum_entropy_reward_training.gif"
sample_animation = save_sample_animation(
    [
        ("Reparameterization", pathwise_history, "tab:blue"),
        ("Log derivative", score_history, "tab:blue"),
    ],
    GIF_PATH,
)
print(f"Saved animation to {GIF_PATH.resolve()}")

from IPython.display import Image, display
display(Image(filename=str(GIF_PATH)))